In [ ]:
# Import all relevant packages

import numpy as np
import pandas as pd
import warnings
import time
from scipy.optimize import minimize
from scipy.special import hyp2f1, gammaln
from arch import arch_model
from fredapi import Fred
import matplotlib
import matplotlib.pyplot as plt
from scipy.stats import norm
matplotlib.use('PDF')

warnings.filterwarnings("ignore")

In [ ]:
"""
CORE FUNCTIONS — ESTIMATION AND INFORMATION CRITERIA:

Defines all functions required for GARCH(1,1) estimation and model comparison.
All functions operate on demeaned percentage-scaled returns (y * 100).

INPUTS (shared across functions):
  - y      — demeaned percentage return series (raw returns * 100)
  - omega  — GARCH intercept parameter
  - alpha  — ARCH coefficient
  - beta_  — GARCH coefficient
  - nu     — tail index parameter (limit-Gaussian and Student-t only)

OUTPUTS (shared across functions):
  - Parameter estimates as lists [omega, alpha, beta] or [omega, alpha, beta, nu]
  - Log-likelihood values (scalar)
  - Information criteria as dict {AIC, BIC, AICc, HQIC}
  - [nan]*3 or [nan]*4 on estimation failure

NOTE ON SCALING:
  All estimation functions expect y on percentage scale (raw returns * 100).
  This improves numerical stability — omega is now on a more tractable scale.
  Alpha and beta are dimensionless and unaffected by scaling.

NOTE ON GRID SEARCH:
  estimate_sub_grid_mle uses a wide fixed grid nu in [1.2, 25] since the true
  nu is unknown in the empirical setting. This differs from the simulation study
  where the grid is centred around the true nu.
"""

def _reconstruct_sigma2(y, omega, alpha, beta_):
    """
    Reconstruct GARCH(1,1) conditional variance path.
    
    Inputs:  y      — demeaned percentage return series
             omega  — GARCH intercept
             alpha  — ARCH coefficient
             beta_  — GARCH coefficient
    Output:  s2     — conditional variance path
    """
    T = len(y)
    s2 = np.zeros(T, float)
    # Initialise at unconditional variance omega/(1-alpha-beta) if stationary,
    # otherwise at sample variance to avoid explosive initialisation
    s2[0] = (omega / max(1e-12, 1 - alpha - beta_)) if (alpha + beta_ < 0.999) else np.var(y)
    for t in range(1, T):
        # GARCH(1,1) recursion: sigma^2_t = omega + alpha*y^2_{t-1} + beta*sigma^2_{t-1}
        # Floor at 1e-12 to prevent numerical issues from negative variance
        s2[t] = max(omega + alpha * y[t-1]**2 + beta_ * s2[t-1], 1e-12)
    return s2


def subgaussian_loglik_fixed_nu(garch_theta, y, nu, penalty_scale=1e10):
    """
    Negative log-likelihood for GARCH(1,1) with limit-Gaussian innovations
    at fixed nu. Soft penalty for support boundary violations.
    
    Inputs:  garch_theta — (omega, alpha, beta)
             y           — demeaned percentage return series
             nu          — fixed tail index
    Output:  negative log-likelihood
    """
    omega, alpha, beta_ = garch_theta
    if (omega <= 1e-12) or (alpha < 0) or (beta_ < 0) or (alpha + beta_ >= 0.999):
        return penalty_scale
    sigma2   = _reconstruct_sigma2(y, omega, alpha, beta_)
    s_e      = nu / np.sqrt((nu+1)*(nu+3))
    
    # Scale standardised residuals by s_e = sigma_epsilon_tilde to obtain
    # the argument of the limit-Gaussian density: z_t = s_e * y_t / sigma_t
    scaled_z = (y / np.sqrt(sigma2)) * s_e
    
    # Check support boundary |z_t| < sqrt(nu)/2
    limit    = np.sqrt(nu) / 2
    outside  = np.abs(scaled_z) >= limit
    
    # Soft quadratic penalty for boundary violations — maintains continuous objective
    if outside.any():
        dist    = np.maximum(0.0, np.abs(scaled_z[outside]) - limit + 1e-6)
        penalty = 1e4 * np.sum(dist**2)
    else:
        penalty = 0.0
    safe_z = scaled_z[~outside]
    if len(safe_z) == 0:
        return penalty_scale
    
    # z_arg = 1 - 4z^2/nu — argument of hypergeometric function in limit-Gaussian density
    z_args   = np.maximum(1 - 4*safe_z**2/nu, 1e-10)
    
    # Log normalising constant — precomputed once per nu for efficiency
    log_norm = (gammaln((nu+1)/2) - 0.5*np.log(np.pi) - gammaln(nu/2)
                - 0.5*np.log(nu) + ((3-nu)/2)*np.log(2))
    log_vals = np.empty(len(safe_z))
    
    # hyp2f1 has no vectorised implementation — loop required
    for i, (z, za) in enumerate(zip(safe_z, z_args)):
        try:
            val = hyp2f1((3-nu)/4, (1-nu)/4, 0.5, za)
            log_vals[i] = (log_norm - 0.5*np.log(za) + np.log(val)
                          if (val > 0 and np.isfinite(val)) else -1e6)
        except:
            log_vals[i] = -1e6
    
    # Log-likelihood: sum of log-densities + log Jacobian terms
    # +len(y)*log(s_e): Jacobian from eps_t -> eps_tilde_t transformation
    # -0.5*sum(log(sigma2)): Jacobian from y_t -> eps_t transformation
    ll       = np.sum(log_vals)
    total_ll = ll - penalty + len(y)*np.log(s_e) - 0.5*np.sum(np.log(sigma2))
    return -total_ll if np.isfinite(total_ll) else penalty_scale


def estimate_qmle(y):
    """
    Gaussian (Q)MLE for GARCH(1,1).
    Input:  y — demeaned percentage return series
    Output: [omega, alpha, beta] or [nan]*3
    """
    try:
        # mean='zero' since y is already demeaned; dist='normal' for Gaussian (Q)MLE
        model = arch_model(y, mean='zero', vol='GARCH', p=1, q=1, dist='normal')
        res   = model.fit(disp='off', show_warning=False)
        return [res.params['omega'], res.params['alpha[1]'], res.params['beta[1]']]
    except:
        # Return NaN on convergence failure
        return [np.nan]*3


def estimate_t_mle(y):
    """
    Student-t (Q)MLE for GARCH(1,1).
    Input:  y — demeaned percentage return series
    Output: [omega, alpha, beta, nu] or [nan]*4
    """
    try:
        # mean='zero' since y is already demeaned; dist='t' for Student-t (Q)MLE
        model = arch_model(y, mean='zero', vol='GARCH', p=1, q=1, dist='t')
        res   = model.fit(disp='off', show_warning=False)
        return [res.params['omega'], res.params['alpha[1]'],
                res.params['beta[1]'], res.params['nu']]
    except:
        # Return NaN on convergence failure
        return [np.nan]*4


def estimate_sub_grid_mle(y, nu_grid=None):
    """
    Limit-Gaussian (Q)MLE for GARCH(1,1) via grid search over nu.
    Wide fixed grid used since true nu unknown in empirical setting.
    Gaussian QMLE warm start with three random restarts per grid point.
    
    Input:  y       — demeaned percentage return series
            nu_grid — optional custom grid; defaults to [1.2, 25] with 20 pts
    Output: [omega, alpha, beta, nu_hat] or [nan]*4
    """
    var_y = np.var(y)
    
    # Wide fixed grid [1.2, 25] — true nu unknown in empirical setting
    # (contrast with simulation study where grid is centred around true nu)
    if nu_grid is None:
        nu_grid = np.linspace(1.2, 25, 20) # Grid should start at 1.2, so search remains in interior of parameter
                                           # space, avoiding boundary nu=1 - consistent with assumption true nu*>1
    best_ll     = -np.inf
    best_params = [np.nan]*4
    constraints = {'type': 'ineq', 'fun': lambda x: 0.999 - x[1] - x[2]}
    bounds      = [(1e-12, None), (1e-6, 0.999), (1e-6, 0.999)]
    qmle = estimate_qmle(y)
    if np.all(np.isfinite(qmle)):
        base_theta = [qmle[0], qmle[1], qmle[2]]
    else:
        base_theta = [var_y * 0.02, 0.10, 0.85]
    for n in nu_grid:
        # Three random restarts per grid point to mitigate local optima
        for _ in range(3):
            theta0    = np.array(base_theta, float)
            # Jitter warm start to explore neighbourhood
            theta0   *= np.random.uniform(0.8, 1.2, size=3)
            theta0[0] = np.clip(theta0[0], 1e-8, var_y * 10)
            theta0[1] = np.clip(theta0[1], 1e-4, 0.3)
            theta0[2] = np.clip(theta0[2], 0.5,  0.95)
            res = minimize(
                subgaussian_loglik_fixed_nu, theta0, args=(y, n),
                method='SLSQP', bounds=bounds, constraints=constraints,
                options={'ftol': 1e-9, 'maxiter': 1000}
            )
            if res.success and -res.fun > best_ll:
                best_ll     = -res.fun
                best_params = [res.x[0], res.x[1], res.x[2], n]
    
    # Filter implausible omega — omega > 50 * sample variance suggests
    # optimiser has inflated baseline to keep residuals within support boundary
    # rather than finding genuine likelihood maximum
    if np.isfinite(best_params[0]) and best_params[0] > 50 * var_y:
        best_params = [np.nan]*4
        
    return best_params


def compute_loglik_gaussian(y, omega, alpha, beta_):
    """Gaussian (Q)MLE log-likelihood at estimated parameters."""
    s2 = _reconstruct_sigma2(y, omega, alpha, beta_)
    # Standard Gaussian log-likelihood: -0.5*(log(2pi) + log(sigma^2) + y^2/sigma^2)
    # -0.5*log(s2) is the Jacobian term from y_t -> eps_t = y_t/sigma_t
    return -0.5 * np.sum(np.log(2*np.pi) + np.log(s2) + y**2/s2)


def compute_loglik_t(y, omega, alpha, beta_, nu):
    """Student-t (Q)MLE log-likelihood at estimated parameters."""
    s2 = _reconstruct_sigma2(y, omega, alpha, beta_)
    z  = y / np.sqrt(s2)
    return np.sum(
        gammaln((nu+1)/2) - gammaln(nu/2) # log normalising constant
        - 0.5*np.log(np.pi*(nu-2))        # scale adjustment for unit variance
        - 0.5*np.log(s2)                  # Jacobian: y_t -> eps_t
        - (nu+1)/2 * np.log(1 + z**2/(nu-2)) # Student-t kernel
    )


def compute_loglik_sub(y, omega, alpha, beta_, nu):
    """Limit-Gaussian (Q)MLE log-likelihood at estimated parameters."""
    # Negate since subgaussian_loglik_fixed_nu returns negative log-likelihood
    return -subgaussian_loglik_fixed_nu([omega, alpha, beta_], y, nu)


def information_criteria(loglik, k, n):
    """
    Compute AIC, BIC, AICc, HQIC.
    
    Inputs:  loglik — maximised log-likelihood
             k      — number of free parameters
             n      — number of observations
    Output:  dict of information criteria
    """
    aic  = 2*k - 2*loglik # penalises by 2 per parameter
    bic  = np.log(n)*k - 2*loglik # stronger penalty for large n
    aicc = aic + (2*k*(k+1))/(n-k-1) if n > k+1 else np.nan # small-sample correction to AIC
    hqic = 2*k*np.log(np.log(n)) - 2*loglik # intermediate penalty
    return {"AIC": aic, "BIC": bic, "AICc": aicc, "HQIC": hqic}

In [ ]:
"""
DATA DOWNLOAD AND PREPARATION:

Downloads five monthly macroeconomic inflation series from the FRED database
and converts to monthly log-difference growth rates.

INPUTS:
  - FRED_API_KEY : free API key from https://fred.stlouisfed.org/docs/api/api_key.html
  - start, end   : sample period for download ("1950-01-01" to "2025-01-01")

OUTPUTS:
  - raw          : pandas DataFrame of raw price level indices (monthly frequency)
  - raw_monthly  : raw resampled to month-end frequency
  - returns      : pandas DataFrame of monthly log-difference growth rates
  - Printed descriptive statistics (skewness, excess kurtosis) for full sample

NOTE ON DATA:
  All series are price level indices obtained from FRED. Series are converted
  to monthly inflation rates via first-order log-differences log(P_t/P_{t-1}),
  which approximates the percentage change in the price level. The full sample
  is downloaded here; the post-2010 subsample is selected in the next cell.

NOTE ON REPRODUCIBILITY:
  Raw data files are not stored in the repository. Running this cell
  reconstructs the datasets used in the thesis by downloading directly
  from FRED using the series identifiers specified in fred_ids.
"""

# Download monthly macroeconomic series from FRED
# Requires a free API key from https://fred.stlouisfed.org/docs/api/api_key.html
FRED_API_KEY = "your_api_key_here"  # replace with your FRED API key
fred = Fred(api_key=FRED_API_KEY)

# Full sample period — post-2010 subsample selected later
start, end = "1950-01-01", "2025-01-01"

# FRED series identifiers for five monthly macroeconomic inflation indicators
fred_ids = {
    "US_PPI":    "PPIACO",             # US Producer Price Index
    "China_CPI": "CHNCPIALLMINMEI",    # China Consumer Price Index
    "US_CPI":    "CPIAUCSL",           # US Consumer Price Index
    "Euro_HICP": "CP0000EZ19M086NEST", # Euro Area Harmonised Index of Consumer Prices
    "UK_CPI":    "GBRCPIALLMINMEI",    # UK Consumer Price Index
}

print("Downloading data from FRED...")
raw = pd.DataFrame()
for name, fid in fred_ids.items():
    try:
        s = fred.get_series(fid, observation_start=start, observation_end=end)
        s.name = name
        raw[name] = s
        print(f"  {name}: {s.first_valid_index().date()} to "
              f"{s.last_valid_index().date()}, n={s.count()}")
        time.sleep(1) # avoid FRED API rate limiting
    except Exception as e:
        print(f"  Failed {name}: {e}")

# Resample to monthly and compute log-differences
# log(P_t/P_{t-1}) gives monthly inflation rate; diff() used if series contains zeros
raw_monthly = raw.resample("ME").last()

def to_growth(series):
    """Convert price level to monthly log-difference growth rate."""
    s = series.dropna()
    if (s <= 0).any():
        return s.diff().dropna() # arithmetic difference (in case of series containing zeros/negatives)
    return np.log(s).diff().dropna() # log-difference (standard case)

returns = pd.DataFrame()
for col in raw_monthly.columns:
    returns[col] = to_growth(raw_monthly[col])

# Full series descriptive statistics — used to motivate post-2010 subsample focus
print("\nFull series descriptive statistics:")
desc_full = pd.DataFrame({
    "N":               returns.count(),
    "Skewness":        returns.skew(),
    "Excess Kurtosis": returns.kurtosis(),
}).round(4)
print(desc_full)

In [ ]:
"""
POST-2010 SUBSAMPLE AND DESCRIPTIVE STATISTICS:

Selects the post-2010 subperiod and computes descriptive statistics for each
series, including standardised residual moments to assess innovation tail behaviour.
Used to produce Table 4.1 in the thesis.

INPUTS:
  - returns      : pandas DataFrame of monthly log-difference growth rates
                   (from previous cell)
  - estimate_qmle, _reconstruct_sigma2 : estimation functions (from Cell 2)

OUTPUTS:
  - post2010_series : dict of five post-2010 monthly return series (N=181 each)
  - desc_df         : DataFrame of descriptive statistics including skewness,
                      excess kurtosis, and standardised residual moments
  - Printed descriptive statistics table

NOTE ON STANDARDISED RESIDUALS:
  Standardised residuals z_t = y_t / sigma_t are computed using Gaussian QMLE
  estimates of sigma_t. Since Gaussian QMLE consistently estimates sigma_t
  regardless of the true innovation distribution, z_t estimates the true
  innovations without imposing distributional assumptions. Their skewness and
  excess kurtosis reveal whether conditional innovations are symmetric and
  light-tailed — the key properties required for the limit-Gaussian specification.
  
NOTE ON SAMPLE PERIOD:
  The post-2010 period represents a structurally distinct macroeconomic
  regime characterised by anchored inflation expectations and reduced
  commodity price volatility following the 2008-09 financial crisis.
  This stable environment provides a natural setting for lighter-tailed
  conditional innovations, consistent with the limit-Gaussian specification.
  The full sample is downloaded in previous cell, but only the post-2010 subsample
  is used for estimation and analysis.
"""

# Post-2010 monthly inflation series
post2010_series = {
    "US PPI":    returns["US_PPI"]["2010":].dropna(),
    "China CPI": returns["China_CPI"]["2010":].dropna(),
    "US CPI":    returns["US_CPI"]["2010":].dropna(),
    "Euro HICP": returns["Euro_HICP"]["2010":].dropna(),
    "UK CPI":    returns["UK_CPI"]["2010":].dropna(),
}

# Compute descriptive statistics including standardised residual moments
# Standardised residuals z_t = y_t / sigma_t (from Gaussian QMLE) strip out
# GARCH volatility clustering, revealing the true innovation tail behaviour
print("POST-2010 DESCRIPTIVE STATISTICS")
print("="*65)

desc_rows = []
for series_name, series in post2010_series.items():
    y_raw = series.dropna().values
    # Demean and scale to percentage returns for numerical stability
    y     = (y_raw - y_raw.mean()) * 100 
    res_q = estimate_qmle(y) 
    if np.all(np.isfinite(res_q)):
        s2    = _reconstruct_sigma2(y, res_q[0], res_q[1], res_q[2])
        z     = y / np.sqrt(s2) # standardised residuals — estimating true innovations
                                # using Gaussian QMLE to get consistent estimate of sigma_t
        kurt_z = pd.Series(z).kurtosis() # excess kurtosis of innovations
        skew_z = pd.Series(z).skew() # skew of innovations
    else:
        kurt_z = np.nan
        skew_z = np.nan
    desc_rows.append({
        "Series":            series_name,
        "N":                 len(y_raw),
        "Skewness":          series.skew(),
        "Excess Kurtosis":   series.kurtosis(),
        "Skewness (std res)":  skew_z,
        "ExKurt (std res)":    kurt_z,
    })

desc_df = pd.DataFrame(desc_rows).set_index("Series").round(4)
print(desc_df)

In [ ]:
"""
GARCH(1,1) ESTIMATION — ALL THREE SPECIFICATIONS:

Fits GARCH(1,1) models under all three innovation specifications to each
post-2010 series and computes information criteria for model comparison.

INPUTS:
  - post2010_series : dict of five post-2010 monthly return series (from Cell 4)
  - estimate_qmle, estimate_t_mle, estimate_sub_grid_mle : estimation functions
  - compute_loglik_gaussian, compute_loglik_t, compute_loglik_sub : log-likelihood functions
  - information_criteria : information criteria function (all from Cell 2)

OUTPUTS:
  - all_emp_results : list of dicts, one per (series, estimator) combination,
                      containing parameter estimates, log-likelihood, and
                      information criteria (AIC, BIC, AICc, HQIC)
  - df_emp          : pandas DataFrame of all_emp_results
  - Printed progress and runtime

NOTE ON REPRODUCIBILITY:
  np.random.seed(42) is set before estimation to ensure the random restarts
  in the limit-Gaussian grid search are reproducible. Unlike the simulation
  study where restart randomness averages out over N=100 replications, each
  series is fitted once here so the seed, and hence restart randomness, must be set explicitly.
"""
# Fix random seed for reproducibility of limit-Gaussian grid search restarts
np.random.seed(42)

all_emp_results = []
start_time = time.time()

for series_name, series in post2010_series.items():
    y_raw = series.dropna().values
    n     = len(y_raw)

    if n < 100:
        print(f"\nSkipping {series_name} — too few observations ({n})")
        continue

    # Demean and scale to percentage returns for numerical stability
    # Omega is then in units of squared percentage returns (divide by 1e4 for raw scale)
    y = (y_raw - y_raw.mean()) * 100

    print(f"\nFitting {series_name} (n={n})...")

    # Gaussian (Q)MLE — 3 parameters (omega, alpha, beta)
    res_q = estimate_qmle(y)
    if np.all(np.isfinite(res_q)):
        ll_q = compute_loglik_gaussian(y, res_q[0], res_q[1], res_q[2])
        ic_q = information_criteria(ll_q, k=3, n=n)
    else:
        ll_q = np.nan
        ic_q = {k: np.nan for k in ["AIC","BIC","AICc","HQIC"]}

    # Student-t (Q)MLE — 4 parameters (omega, alpha, beta, nu)
    res_t = estimate_t_mle(y)
    if np.all(np.isfinite(res_t)):
        ll_t = compute_loglik_t(y, res_t[0], res_t[1], res_t[2], res_t[3])
        ic_t = information_criteria(ll_t, k=4, n=n)
    else:
        ll_t = np.nan
        ic_t = {k: np.nan for k in ["AIC","BIC","AICc","HQIC"]}

    # Limit-Gaussian (Q)MLE — 4 parameters (omega, alpha, beta, nu)
    # Grid search over nu in [1.2, 25]; wide search grid 
    # since true nu is unknown in the empirical setting
    res_s = estimate_sub_grid_mle(y)
    if np.all(np.isfinite(res_s)):
        ll_s = compute_loglik_sub(y, res_s[0], res_s[1], res_s[2], res_s[3])
        ic_s = information_criteria(ll_s, k=4, n=n)
    else:
        ll_s = np.nan
        ic_s = {k: np.nan for k in ["AIC","BIC","AICc","HQIC"]}

    elapsed = (time.time() - start_time) / 60
    print(f"  Done ({elapsed:.1f} min elapsed)")

    # Store results — NaN for nu where not applicable (Gaussian QMLE)
    for dist, res, ic, ll in [
        ("Gaussian (Q)MLE",       res_q+[np.nan], ic_q, ll_q),
        ("Student-t (Q)MLE",      res_t,          ic_t, ll_t),
        ("Limit-Gaussian (Q)MLE", res_s,          ic_s, ll_s),
    ]:
        all_emp_results.append({
            "Series":    series_name,
            "Estimator": dist,
            "omega":     res[0] if np.isfinite(res[0]) else np.nan,
            "alpha":     res[1] if np.isfinite(res[1]) else np.nan,
            "beta":      res[2] if np.isfinite(res[2]) else np.nan,
            "nu":        res[3] if len(res)>3 and np.isfinite(res[3]) else np.nan,
            "loglik":    ll,
            **ic
        })

df_emp = pd.DataFrame(all_emp_results)
print(f"\nTotal runtime: {(time.time()-start_time)/60:.1f} minutes")

In [ ]:
"""
RESULTS TABLES — PARAMETER ESTIMATES AND INFORMATION CRITERIA:

Prints parameter estimates and information criteria for all series and
estimators, used to produce Tables 4.2 and 4.3 in the thesis.

INPUTS:
  - df_emp : pandas DataFrame of estimation results (from previous cell)

OUTPUTS:
  - Printed parameter estimates table: omega, alpha, beta, nu, log-likelihood
  - Printed information criteria comparison tables: AIC, BIC, AICc, HQIC
    (one pivot table per criterion, rows = series, columns = estimator)
"""

print("PARAMETER ESTIMATES")
print("="*65)
print(df_emp[["Series","Estimator","omega","alpha","beta",
              "nu","loglik"]].round(4).to_string())

for criterion in ["AIC", "BIC", "AICc", "HQIC"]:
    print(f"\n{criterion} COMPARISON (lower is better)")
    print("="*65)
    print(df_emp.pivot_table(
        index="Series", columns="Estimator", values=criterion
    ).round(4))

In [ ]:
"""
DISTRIBUTION HISTOGRAMS:

Generates histograms of monthly log-difference distributions for each series,
overlaid with a fitted Gaussian density as a visual benchmark for tail behaviour.
Used to produce Figure 4.1 in the thesis.

INPUTS:
  - post2010_series : dict of five post-2010 monthly return series (from Cell 4)

OUTPUTS:
  - Five PDF files saved to working directory:
      US_PPI_post2010_histogram.pdf
      China_CPI_post2010_histogram.pdf
      US_CPI_post2010_histogram.pdf
      Euro_HICP_post2010_histogram.pdf
      UK_CPI_post2010_histogram.pdf
  Each histogram shows the empirical distribution of monthly percentage
  log-differences with skewness and excess kurtosis reported in the legend,
  overlaid with a Gaussian density fitted to the same mean and standard deviation.
"""

print("Generating histograms...")

for series_name, series in post2010_series.items():
    y_pct  = series.dropna() * 100 # Convert to percentage for display (improves readability)
    mu, sd = y_pct.mean(), y_pct.std()

    fig, ax = plt.subplots(figsize=(8, 5))
    # Histogram with skewness and excess kurtosis reported in legend
    ax.hist(y_pct, bins=30, density=True, alpha=0.6,
            color='steelblue', edgecolor='white',
            label=f'Skew={series.skew():.3f}, ExKurt={series.kurtosis():.3f}')
    # Fitted Gaussian density with same mean and std as data — visual benchmark
    x = np.linspace(y_pct.quantile(0.005), y_pct.quantile(0.995), 400)
    ax.plot(x, norm.pdf(x, mu, sd), 'r-', lw=2, label='Fitted Normal')
    ax.set_xlabel('Monthly log-difference (%)', fontsize=12)
    ax.set_ylabel('Density', fontsize=12)
    ax.legend(fontsize=10)
    plt.tight_layout()
    fname = series_name.replace(" ", "_")
    plt.savefig(f'{fname}_post2010_histogram.pdf', bbox_inches='tight')
    plt.close()
    print(f"  Saved {fname}_post2010_histogram.pdf")

print("Done.")